## **AdaBoost:**

While **Gradient Boosting** focuses on predicting **residuals** (errors), **AdaBoost** focuses on re-weighting **data points**. 

In AdaBoost, you don't change the prediction values themselves; instead, you change how much "importance" each row in your dataset has.

### The Core Difference in Logic

| Feature | Gradient Boosting (GBM) | AdaBoost |
| :--- | :--- | :--- |
| **Target of Update** | The **prediction value** ($\hat{y}$) is updated by adding a new model. | The **sample weight** ($w$) is updated to make hard cases more important. |
| **What the next tree sees** | A new target variable (the residuals/gradients). | The same $y$ values, but some rows are "heavier" than others. |
  | **Mathematical Focus** | Minimizing a Loss Function via Gradient Descent. | Minimizing a Weighted Error Rate. |

---

### How AdaBoost Works (Step-by-Step)

#### 1. Initialization
Every data point starts with the same weight: $w_i = \frac{1}{N}$ (where $N$ is the number of samples). Every sample is equally important at the start.

#### 2. The Iterative Loop
For each iteration (each new "Stump"):

*   **Step A: Train on Weighted Data:** A weak learner (usually a Decision Stump) is trained. However, it doesn't just look at $y$; it looks at $(y, w)$. It tries to minimize the **weighted error**. 
    *   *If a point has a high weight, misclassifying it results in a huge penalty.*
*   **Step B: Calculate Error ($\epsilon$):** We calculate the error of this stump, but only summing the weights of the **misclassified** points.
    $$\epsilon = \frac{\text{Sum of weights of misclassified points}}{\text{Total sum of all weights}}$$
*   **Step C: Calculate Model Importance ($\alpha$):** We calculate how much "say" this specific stump should have in the final vote. 
    $$\alpha = \frac{1}{2} \ln\left(\frac{1 - \epsilon}{\epsilon}\right)$$
    *   If error is low, $\alpha$ is high (the model is trusted).
    *   If error is $0.5$ (random guessing), $\alpha$ is $0$.
*   **Step D: Update Weights ($w$):** This is the "Adaptive" part. We update the weights for every point:
    *   **For correctly classified points:** Decrease their weight.
    *   **For misclassified points:** Increase their weight.
    $$w_{new} = w_{old} \cdot e^{\pm \alpha}$$

#### 3. Final Prediction
The final model is a **weighted sum** of all the stumps:
$$\text{Final Prediction} = \text{sign}\left( \sum_{t=1}^{T} \alpha_t \cdot h_t(x) \right)$$

---

### A Simple Analogy: The Student and the Tutor

*   **Gradient Boosting (The Refiner):** Imagine a student solving math problems. After each problem, the teacher says, *"You were off by +5 in your answer; try to adjust your logic to fix that specific +5 error next time."* (Focus on **correcting the value**).
*   **AdaBoost (The Focused Tutor):** Imagine a student solving a set of flashcards. After each round, the teacher says, *"You keep getting the 'Geometry' cards wrong. From now on, we are going to study the 'Geometry' cards 10 times more intensely than the others."* (Focus on **re-weighting the importance**).

### Summary Table: Residuals vs. Weights

| Concept | Gradient Boosting (GBM) | AdaBoost |
| :--- | :--- | :--- |
| **What is "New"?** | The target $y$ changes to $(y - \hat{y})$. | The weight $w$ of the rows changes. |
| **The Learner's Goal** | Predict the remaining error. | Correctly classify the "heavy" points. |
| **Complexity** | Can handle complex loss functions (LogLoss, MSE). | Primarily used for classification (minimizing error rate). |

### **1) Using Scikit-learn:**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
# generate synthetic data
X, y = make_classification(n_samples=100, n_features=20, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# initialize the base learner
base_estimator = DecisionTreeClassifier(max_depth=1)

# initialize AdaBoost
ada = AdaBoostClassifier(estimator=base_estimator, n_estimators=100, learning_rate=1.0, random_state=42)

# train
ada.fit(X_train, y_train)

# predict and evaluate
predictions = ada.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f'AdaBoost Accuracy: {accuracy:.4f}')

AdaBoost Accuracy: 0.9200


### **2) AdaBoost - From Scratch**

In [3]:
class SimpleAdaBoost:
    def __init__(self, n_estimators=10, max_depth=1):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.alphas = []
        self.models = []
        self.weights = None
    
    def fit(self, X, y):
        n_samples = X.shape[0]
        self.weights = np.full(n_samples, 1 / n_samples) # initialize the weights

        for _ in range(self.n_estimators):
            tree = DecisionTreeClassifier(max_depth=self.max_depth)
            tree.fit(X, y, sample_weight=self.weights)

            # calculate the error
            predictions = tree.predict(X)
            
            misclassified = predictions != y
            error = np.sum(self.weights[misclassified]) / np.sum(self.weights)

            # calculate the alpha
            error = max(1e-10, min(error, 1 - 1e-10)) # avoid division as 0
            alpha = 1/2 * np.log((1 - error) / error)

            # update the weights
            self.weights *= np.exp(alpha * misclassified) # increase weight for misclassified points
            self.weights *= np.exp(-alpha * (~misclassified)) # decrease weight for correct classified points

            self.weights / np.sum(self.weights) # rescale weights so sum to 1

            # store the model and the alpha
            self.models.append(tree)
            self.alphas.append(alpha)

            print(f'Tree {len(self.models)}: Error={error:.4f}, Alpha={alpha:.4f}')
        
    def predict(self, X):
        final_preds = np.zeros(X.shape[0])

        for alpha, tree in zip(self.alphas, self.models):
            tree_pred = np.where(tree.predict(X) == 1, 1, -1) # two classes model
            final_preds += alpha * tree_pred # weighted sum of all tree prediction
        
        return np.where(final_preds >= 0, 1, 0)



In [4]:
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=100, n_features=5, random_state=42)

# Initialize and train
booster = SimpleAdaBoost(n_estimators=10)
booster.fit(X, y)

# Predict
y_pred = booster.predict(X)
accuracy = np.mean(y_pred == y)
print(f"\nFinal Scratch AdaBoost Accuracy: {accuracy * 100:.2f}%")

Tree 1: Error=0.0400, Alpha=1.5890
Tree 2: Error=0.1458, Alpha=0.8838
Tree 3: Error=0.1424, Alpha=0.8977
Tree 4: Error=0.1386, Alpha=0.9133
Tree 5: Error=0.1921, Alpha=0.7182
Tree 6: Error=0.1423, Alpha=0.8982
Tree 7: Error=0.1541, Alpha=0.8512
Tree 8: Error=0.2956, Alpha=0.4343
Tree 9: Error=0.1911, Alpha=0.7214
Tree 10: Error=0.1728, Alpha=0.7828

Final Scratch AdaBoost Accuracy: 100.00%
